# Lab: Drift Detection with Evidently AI

> **Ops Context:** This lab is your monitoring stack setup for ML models.
> By the end, you'll know how to detect when a model is degrading —
> even when the pipeline is green and all health checks pass.

## What We're Building
- A simulated "training" dataset and a "production" dataset with drift injected
- Data drift reports using Evidently AI
- Prediction drift detection
- A concept drift simulation

## Key Insight Before We Start
Traditional monitoring asks: "Is the service UP?"
ML monitoring asks: "Is the service CORRECT?"

A model can return 200 OK on every request while being completely wrong.
This lab teaches you to catch that.

## Setup
Run `setup.sh` first, then activate the venv:
```bash
source .venv/bin/activate
jupyter notebook lab-drift-detection.ipynb
```

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric

np.random.seed(42)
print("All imports successful. Let's detect some drift.")

## Part 1: Create Our "Training" Dataset

> **Ops Translation:** This is our baseline — like capturing your system's normal
> behavior during a quiet period. We'll use this as reference to detect anomalies later.

We're simulating a **fraud detection** scenario at a bank:
- Features: transaction amount, time of day, merchant category, transaction velocity
- Target: is_fraud (1 = fraud, 0 = legit)

In [ ]:
def generate_training_data(n_samples=5000):
    """Generate our 'training' distribution — the world as the model knows it."""
    data = pd.DataFrame({
        'transaction_amount': np.random.lognormal(mean=3.5, sigma=1.0, size=n_samples),
        'hour_of_day': np.random.choice(range(24), size=n_samples, 
                                         p=[0.02]*6 + [0.06]*12 + [0.04]*6),
        'merchant_category': np.random.choice(
            ['retail', 'food', 'travel', 'online', 'atm'], 
            size=n_samples, p=[0.3, 0.25, 0.15, 0.2, 0.1]),
        'transaction_velocity_24h': np.random.poisson(lam=3, size=n_samples),
    })
    
    # Create fraud labels based on known patterns
    fraud_probability = (
        (data['transaction_amount'] > 200).astype(float) * 0.15 +
        (data['hour_of_day'].isin([0,1,2,3,4])).astype(float) * 0.2 +
        (data['transaction_velocity_24h'] > 8).astype(float) * 0.3 +
        (data['merchant_category'] == 'online').astype(float) * 0.1
    )
    data['is_fraud'] = (np.random.random(n_samples) < fraud_probability).astype(int)
    
    return data

training_data = generate_training_data()
print(f"Training data shape: {training_data.shape}")
print(f"Fraud rate: {training_data['is_fraud'].mean():.2%}")
print(f"\nFeature distributions (training/baseline):")
print(training_data.describe().round(2))

## Part 2: Train a Simple Model

> **Ops Translation:** We're building the service that will go to production.
> The model learns patterns from the training data. Everything after this
> simulates what happens when production reality diverges from training reality.

In [ ]:
# Encode categorical feature
training_encoded = pd.get_dummies(training_data, columns=['merchant_category'])

X = training_encoded.drop('is_fraud', axis=1)
y = training_encoded['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest (simple, interpretable enough for our purposes)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Baseline performance
y_pred = model.predict(X_test)
print("=== BASELINE MODEL PERFORMANCE ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
print("\n--- Operator Note ---")
print("This is our 'known good' state. Like a baseline Grafana snapshot.")
print("Everything below simulates production divergence from this baseline.")

## Part 3: Simulate Production Data WITH Data Drift

> **Ops Translation:** Time passes. The real world changes. New customer behaviors
> emerge. This is like when your traffic patterns shift after a product launch
> but your alerting thresholds are still set for the old normal.

We'll inject **data drift** by changing the input distributions:
- Transaction amounts increase (inflation + new high-value products)
- Transaction velocity increases (new mobile app drives more frequent transactions)
- Merchant category distribution shifts (online shopping grows)

In [ ]:
def generate_production_data_with_drift(n_samples=5000):
    """Generate 'production' data where input distributions have shifted."""
    data = pd.DataFrame({
        # DRIFT: amounts are higher (inflation + new products)
        'transaction_amount': np.random.lognormal(mean=4.0, sigma=1.2, size=n_samples),
        # DRIFT: more late-night transactions (new global customer base)
        'hour_of_day': np.random.choice(range(24), size=n_samples,
                                         p=[0.05]*6 + [0.04]*12 + [0.05]*6),
        # DRIFT: online category grew significantly
        'merchant_category': np.random.choice(
            ['retail', 'food', 'travel', 'online', 'atm'],
            size=n_samples, p=[0.2, 0.15, 0.1, 0.4, 0.15]),
        # DRIFT: higher transaction velocity (mobile app launched)
        'transaction_velocity_24h': np.random.poisson(lam=6, size=n_samples),
    })
    
    # Same fraud logic as training (concept hasn't drifted yet)
    fraud_probability = (
        (data['transaction_amount'] > 200).astype(float) * 0.15 +
        (data['hour_of_day'].isin([0,1,2,3,4])).astype(float) * 0.2 +
        (data['transaction_velocity_24h'] > 8).astype(float) * 0.3 +
        (data['merchant_category'] == 'online').astype(float) * 0.1
    )
    data['is_fraud'] = (np.random.random(n_samples) < fraud_probability).astype(int)
    
    return data

production_data = generate_production_data_with_drift()
print(f"Production data shape: {production_data.shape}")
print(f"Production fraud rate: {production_data['is_fraud'].mean():.2%}")
print(f"Training fraud rate was: {training_data['is_fraud'].mean():.2%}")
print(f"\n--- What changed (operator view) ---")
print(f"Avg transaction amount: training={training_data['transaction_amount'].mean():.0f} → prod={production_data['transaction_amount'].mean():.0f}")
print(f"Avg velocity: training={training_data['transaction_velocity_24h'].mean():.1f} → prod={production_data['transaction_velocity_24h'].mean():.1f}")
print(f"Online %: training={( training_data['merchant_category']=='online').mean():.1%} → prod={(production_data['merchant_category']=='online').mean():.1%}")

## Part 4: Generate Data Drift Report with Evidently

> **Ops Translation:** This is like running a diff between your baseline config
> and current state. Evidently does statistical tests on each feature to determine
> if the distribution has significantly changed.

Key things to look for in the report:
- **Dataset Drift:** Did the overall dataset drift? (yes/no)
- **Per-feature drift:** Which specific features drifted?
- **Drift score:** How severe is the drift?

In [ ]:
# Prepare data for Evidently (needs same columns)
reference = training_data.copy()
current = production_data.copy()

# Generate the Data Drift Report
data_drift_report = Report(metrics=[
    DataDriftPreset(),
])

data_drift_report.run(
    reference_data=reference,
    current_data=current
)

# Display the report
data_drift_report

# Operator note: In production, you'd save this to HTML and serve it
# or extract metrics to push to Prometheus/Grafana
# data_drift_report.save_html("drift_report.html")

### Interpreting the Data Drift Report

**What to look for (operator checklist):**

1. **Dataset Drift = Yes** → At least some features have shifted significantly
2. **Which features drifted?** → These tell you WHERE the world changed
3. **Drift detection method** → Evidently uses statistical tests (KS, chi-squared) automatically
4. **p-value** → Lower = more confident the drift is real (not random noise)

> **Critical insight:** The pipeline is still GREEN. All health checks pass.
> The model still returns predictions. But the inputs have shifted.
> This is the gap between "service is up" and "service is correct."

Let's also look at individual feature drift:

In [ ]:
# Detailed per-feature drift report
feature_drift_report = Report(metrics=[
    ColumnDriftMetric(column_name='transaction_amount'),
    ColumnDriftMetric(column_name='hour_of_day'),
    ColumnDriftMetric(column_name='transaction_velocity_24h'),
    ColumnDriftMetric(column_name='merchant_category'),
])

feature_drift_report.run(
    reference_data=reference,
    current_data=current
)

feature_drift_report

## Part 5: Prediction Drift — The First Symptom You'll See

> **Ops Translation:** This is your error rate / latency p99 equivalent.
> You don't need to know WHY yet — you just see the OUTPUT changing.
> Prediction drift is often the first alert that fires.

Let's run our model on both datasets and compare the prediction distributions:

In [ ]:
# Get predictions on training data (baseline output distribution)
train_encoded = pd.get_dummies(training_data.drop('is_fraud', axis=1), 
                                columns=['merchant_category'])
# Ensure columns match
for col in X_train.columns:
    if col not in train_encoded.columns:
        train_encoded[col] = 0
train_encoded = train_encoded[X_train.columns]

train_predictions = model.predict_proba(train_encoded)[:, 1]

# Get predictions on production data (current output distribution)
prod_encoded = pd.get_dummies(production_data.drop('is_fraud', axis=1),
                               columns=['merchant_category'])
for col in X_train.columns:
    if col not in prod_encoded.columns:
        prod_encoded[col] = 0
prod_encoded = prod_encoded[X_train.columns]

prod_predictions = model.predict_proba(prod_encoded)[:, 1]

# Compare prediction distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_predictions, bins=50, alpha=0.7, color='blue', label='Training')
axes[0].set_title('Prediction Distribution — Training (Baseline)')
axes[0].set_xlabel('Fraud Probability Score')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].axvline(x=0.5, color='red', linestyle='--', label='Decision Threshold')

axes[1].hist(prod_predictions, bins=50, alpha=0.7, color='orange', label='Production')
axes[1].set_title('Prediction Distribution — Production (Drifted)')
axes[1].set_xlabel('Fraud Probability Score')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].axvline(x=0.5, color='red', linestyle='--', label='Decision Threshold')

plt.tight_layout()
plt.savefig('prediction_drift_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Quantify the shift
print("=== PREDICTION DRIFT SUMMARY ===")
print(f"Training: mean score = {train_predictions.mean():.3f}, % flagged = {(train_predictions > 0.5).mean():.2%}")
print(f"Production: mean score = {prod_predictions.mean():.3f}, % flagged = {(prod_predictions > 0.5).mean():.2%}")
print(f"\n--- Operator Alert ---")
print(f"Prediction rate shifted from {(train_predictions > 0.5).mean():.2%} to {(prod_predictions > 0.5).mean():.2%}")
print("This would trigger a prediction drift alert in production monitoring.")

### Evidently Target/Prediction Drift Report

Let's use Evidently's built-in target drift analysis:

In [ ]:
# Add predictions as a column for Evidently analysis
reference_with_preds = reference.copy()
reference_with_preds['prediction'] = train_predictions

current_with_preds = current.copy()
current_with_preds['prediction'] = prod_predictions

# Target Drift Report
target_drift_report = Report(metrics=[
    TargetDriftPreset(),
])

target_drift_report.run(
    reference_data=reference_with_preds,
    current_data=current_with_preds
)

target_drift_report

## Part 6: "Pipeline is Green" vs "Model is Degrading"

> **This is THE key lesson for an ops leader.**
> 
> Your CI/CD passed. Your health checks are green. Latency is normal.
> CPU/memory look fine. No errors in logs. But the model is WRONG.

Let's demonstrate this clearly:

In [ ]:
# Simulate what your traditional monitoring sees
print("=" * 60)
print("TRADITIONAL OPS DASHBOARD")
print("=" * 60)
print(f"  Service health:      ✅ HEALTHY")
print(f"  Endpoint latency:    ✅ p99 = 45ms (threshold: 200ms)")
print(f"  Error rate:          ✅ 0.01% (threshold: 1%)")
print(f"  CPU utilization:     ✅ 34% (threshold: 80%)")
print(f"  Memory:              ✅ 2.1GB / 4GB")
print(f"  Pod restarts:        ✅ 0 in last 24h")
print(f"  Last deploy:         ✅ 3 days ago, no rollbacks")
print()
print("=" * 60)
print("ML MODEL MONITORING (what you're MISSING without drift detection)")
print("=" * 60)

# Evaluate model on production data
prod_binary_preds = model.predict(prod_encoded)
prod_accuracy = accuracy_score(production_data['is_fraud'], prod_binary_preds)
train_accuracy = accuracy_score(y_test, y_pred)

print(f"  Baseline accuracy:   {train_accuracy:.3f}")
print(f"  Current accuracy:    {prod_accuracy:.3f}")
print(f"  Accuracy drop:       ⚠️  {train_accuracy - prod_accuracy:.3f}")
print(f"  Data drift detected: ⚠️  YES (3 of 4 features drifted)")
print(f"  Prediction drift:    ⚠️  YES (flag rate shifted significantly)")
print()
print("--- OPERATOR TAKEAWAY ---")
print("Without ML monitoring, this model silently degrades.")
print("Your SRE instincts are right: if you can't measure it, you can't manage it.")
print("Data drift + prediction drift = leading indicators of model failure.")

## Part 7: Simulating Concept Drift

> **Ops Translation:** The rules of the game changed, but nobody told the model.
> 
> This is the hardest drift to detect because the DATA might look the same,
> but what counts as "fraud" has fundamentally changed.

**Scenario:** A new legitimate fintech integration causes high-velocity small 
transactions that LOOK like fraud patterns but are actually legit. The concept 
of "fraud" has drifted — old patterns that meant fraud now mean normal usage.

In [ ]:
def generate_concept_drift_data(n_samples=5000):
    """
    Generate data where the RELATIONSHIP between features and target changed.
    
    The input distributions are SIMILAR to training (no data drift),
    but what constitutes fraud has changed.
    """
    data = pd.DataFrame({
        # Same distributions as training (no data drift!)
        'transaction_amount': np.random.lognormal(mean=3.5, sigma=1.0, size=n_samples),
        'hour_of_day': np.random.choice(range(24), size=n_samples,
                                         p=[0.02]*6 + [0.06]*12 + [0.04]*6),
        'merchant_category': np.random.choice(
            ['retail', 'food', 'travel', 'online', 'atm'],
            size=n_samples, p=[0.3, 0.25, 0.15, 0.2, 0.1]),
        'transaction_velocity_24h': np.random.poisson(lam=3, size=n_samples),
    })
    
    # CONCEPT DRIFT: The rules changed!
    # High velocity is no longer a fraud signal (legit fintech usage)
    # Instead, fraud now correlates with: specific amount ranges + travel category
    fraud_probability = (
        (data['transaction_amount'].between(100, 300)).astype(float) * 0.25 +
        (data['merchant_category'] == 'travel').astype(float) * 0.3 +
        (data['hour_of_day'].isin([10,11,14,15])).astype(float) * 0.15
        # NOTE: transaction_velocity_24h is NO LONGER a fraud indicator!
    )
    data['is_fraud'] = (np.random.random(n_samples) < fraud_probability).astype(int)
    
    return data

concept_drift_data = generate_concept_drift_data()
print(f"Concept drift data shape: {concept_drift_data.shape}")
print(f"Concept drift fraud rate: {concept_drift_data['is_fraud'].mean():.2%}")
print(f"\n--- What changed (invisible to data drift detection!) ---")
print("Old fraud rule: high amount + late night + high velocity + online")
print("New fraud rule: mid amount + business hours + travel")
print("The FEATURES look the same. The MEANING changed.")

In [ ]:
# Run data drift detection on concept-drifted data
concept_reference = training_data.copy()
concept_current = concept_drift_data.copy()

# Check for data drift
concept_drift_report = Report(metrics=[
    DataDriftPreset(),
])

concept_drift_report.run(
    reference_data=concept_reference.drop('is_fraud', axis=1),
    current_data=concept_current.drop('is_fraud', axis=1)
)

print("=== DATA DRIFT CHECK ON CONCEPT-DRIFTED DATA ===")
print("(Look at the report below — data drift should be MINIMAL)")
print()
print("This is why concept drift is dangerous:")
print("- Data drift detection says: 'Everything looks normal'")  
print("- But the model is making wrong predictions because")
print("  the RELATIONSHIP between features and target changed.")
print()
concept_drift_report

In [ ]:
# Now evaluate the model on concept-drifted data
concept_encoded = pd.get_dummies(concept_drift_data.drop('is_fraud', axis=1),
                                  columns=['merchant_category'])
for col in X_train.columns:
    if col not in concept_encoded.columns:
        concept_encoded[col] = 0
concept_encoded = concept_encoded[X_train.columns]

concept_preds = model.predict(concept_encoded)
concept_accuracy = accuracy_score(concept_drift_data['is_fraud'], concept_preds)

print("=== MODEL PERFORMANCE UNDER CONCEPT DRIFT ===")
print(f"Baseline accuracy (training):     {train_accuracy:.3f}")
print(f"Accuracy under data drift:        {prod_accuracy:.3f}")
print(f"Accuracy under concept drift:     {concept_accuracy:.3f}")
print()
print("Confusion Matrix (concept drift):")
print(confusion_matrix(concept_drift_data['is_fraud'], concept_preds))
print()
print(classification_report(concept_drift_data['is_fraud'], concept_preds, 
                           target_names=['Legit', 'Fraud']))
print("--- OPERATOR INSIGHT ---")
print("Notice: The model's accuracy DROPPED significantly,")
print("but data drift detection didn't catch it!")
print("This is why you need GROUND TRUTH monitoring (actual outcomes)")
print("in addition to data drift monitoring.")
print()
print("Detection strategy for concept drift:")
print("1. Compare predictions vs actual outcomes (delayed feedback)")
print("2. Monitor business metrics (fraud losses, false positive complaints)")
print("3. Regular model revalidation against fresh labeled data")

## Part 8: Summary — The Complete Picture

| Scenario | Data Drift Detected? | Model Accuracy | Alert Fires? | Root Cause |
|----------|---------------------|----------------|-------------|-----------|
| Baseline (training) | No | High | No | N/A |
| Data drift (Part 3-5) | **YES** | Degraded | YES (data + prediction drift) | Input distributions shifted |
| Concept drift (Part 7) | **NO** | Severely degraded | Only if monitoring ground truth | Underlying relationship changed |

### The Monitoring Stack You Need (in priority order):

1. **Prediction drift** — fast, no labels needed, your canary
2. **Data drift** — catches input distribution changes
3. **Ground truth monitoring** — catches concept drift (delayed but critical)
4. **Pipeline validation** — catches training-serving skew (your team's domain)

## 🎯 Milestone 1: Write Your ML Failure Modes Runbook

> **Your task:** Using what you've learned in this lab and the README, write a
> 1-page ML failure modes runbook. Use the template in `milestone-1-template.md`.
> 
> Cover these failure modes:
> 1. Data drift detected, no accuracy drop yet
> 2. Data drift detected WITH accuracy drop
> 3. Concept drift suspected (accuracy dropping, no data drift)
> 4. Training-serving skew after a pipeline deploy
> 5. Sudden prediction drift (unknown root cause)
>
> For each one, fill in:
> - Detection method (what alert fires?)
> - Severity classification  
> - Immediate response
> - Root cause investigation steps
> - Resolution options
> - Prevention measures
>
> **Why this matters for your role:** As ML support team lead, your team will be
> the first responders to these alerts. This runbook is what you'll hand them
> on day one. Write it like you'd write an incident runbook for a production service.
>
> **Stretch goal:** Map each failure mode to a hypothetical PagerDuty alert
> with severity, escalation path, and SLA for response/resolution.

In [ ]:
print("=" * 60)
print("LAB COMPLETE")
print("=" * 60)
print()
print("Key takeaways:")
print("1. Data drift = inputs changed (detectable with Evidently)")
print("2. Prediction drift = outputs changed (your first-line alert)")
print("3. Concept drift = rules changed (hardest to catch, needs ground truth)")
print("4. Traditional monitoring WILL NOT catch model degradation")
print("5. You need a layered monitoring approach (like defense in depth)")
print()
print("Next step: Complete Milestone 1 using milestone-1-template.md")
print()
print("Optional: Save the drift report to HTML for reference:")
print("  data_drift_report.save_html('data_drift_report.html')")